> ### 💾 Submitting Your Work — Read This First
>
> **Villanova does not use Google Workspace**, so Colab will NOT auto-save your changes.
>
> **To submit this lab:**
> 1. Work through the notebook in Colab.
> 2. When you're done, go to **File → Download → Download .ipynb**.
> 3. Upload the downloaded `.ipynb` to the Lab 13 dropbox on **Brightspace**.
>
> **Save often during the lab** so you don't lose work if your tab closes.


# Lab 13: Can We Predict How Well a Radio Station Comes In?
## Building a Signal Strength Model with Real FCC Data

**CSC 2053 — Fall 2026**

---

You've spent the last few weeks analyzing radio station data with Pandas and NumPy.
Today you're going to use it to *predict* something real: **how strong a radio signal
is at a specific receiving location.**

The FCC measures field strength in **dBu** (decibels above one microvolt per meter).
Higher = clearer reception. Lower = static.

Your goal: build a machine learning model that predicts field strength from what we
know about the transmitter — its power, height, frequency, and distance from the receiver.

This is called **regression** — predicting a continuous number, not a category.

---

*The data loading and longitude fix are the same patterns you used in Lab 11.*
*The new piece today is `scikit-learn`.*


---
## Setup

Two new libraries today: **seaborn** (for quick, pretty plots) and **scikit-learn**
(Python's standard machine learning toolkit). You already have pandas, numpy, matplotlib,
and folium from earlier labs.

Run this cell first, then wait for "Ready." before moving on.


In [ ]:
# Run this first
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "folium", "-q"])
subprocess.run([sys.executable, "-m", "pip", "install", "seaborn", "-q"])
subprocess.run([sys.executable, "-m", "pip", "install", "scikit-learn", "-q"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from IPython.display import display

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

sns.set_theme(style="whitegrid")
print("Ready.")


---
## Load the Data

1,510 FM stations measured from 25 receiver locations across the US.
Each row is one transmitter–receiver pair with a measured field strength.


In [ ]:
URL = "https://raw.githubusercontent.com/CSC-2053-100-Fall25/python-ml-template/main/fm_stations_25_locations_ml_dataset.csv"
df = pd.read_csv(URL)
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
df[['callsign','city','search_location','distance','erp','haat','frequency','field_strength']].head(8)


### Fix Transmitter Longitude — Same as Lab 11

The transmitter `lon` column is stored as a positive number.
Negate it so coordinates are correct for the Western Hemisphere.
(The receiver `search_lon` is already negative — no fix needed there.)


In [ ]:
df['lon'] = df['lon'] * -1
print("Sample transmitter lons after fix:", df['lon'].head(3).tolist())


### Clean — Drop Rows with Missing Key Values

"Garbage in, garbage out." Models can't learn from rows where
the features or target are missing. We keep only rows where all
five key columns have valid, positive values.


In [ ]:
KEY = ['distance', 'erp', 'frequency', 'haat', 'field_strength']
df_clean = df.dropna(subset=KEY).copy()
df_clean = df_clean[(df_clean[KEY] > 0).all(axis=1)].reset_index(drop=True)
print(f"Rows after cleaning: {len(df_clean):,}  (removed {len(df) - len(df_clean)})")


---
## Plotting Primer — Meet matplotlib and seaborn

**Rule #1 in ML: understand your data before you model it.** Before we build anything,
we're going to look at this radio dataset three ways with two libraries you'll see
everywhere in Python data work:

- **matplotlib** (`plt`) — the OG plotting library. Verbose, but gives you total control.
- **seaborn** (`sns`) — built on top of matplotlib. One-line plots that are pretty by default,
  and knows how to work directly with a DataFrame.

Use seaborn when you're exploring a new dataset; use matplotlib when you need
fine-grained control.


### Three ways to draw the same histogram

Below are three equivalent histograms of `field_strength`. Run all three and compare
the code length, the look, and the amount of setup each one needs.


In [ ]:
# Way 1 — pandas' built-in .hist() (shortest)
df_clean['field_strength'].hist(bins=30, figsize=(7, 3), color='steelblue', edgecolor='white')
plt.title('pandas .hist()')
plt.xlabel('Field Strength (dBu)')
plt.show()


In [ ]:
# Way 2 — matplotlib directly (most control)
plt.figure(figsize=(7, 3))
plt.hist(df_clean['field_strength'], bins=30, color='steelblue', edgecolor='white')
plt.xlabel('Field Strength (dBu)')
plt.ylabel('Count')
plt.title('matplotlib plt.hist')
plt.tight_layout()
plt.show()


In [ ]:
# Way 3 — seaborn (prettiest default, DataFrame-aware)
plt.figure(figsize=(7, 3))
sns.histplot(data=df_clean, x='field_strength', bins=30, color='steelblue')
plt.title('seaborn sns.histplot')
plt.show()


**Quick reflection — which do you prefer?** No wrong answer. Notice seaborn didn't need
you to pass `df_clean['field_strength']` — just the DataFrame and the column name.

---

### Numbers behind the picture

Before you move on, take a look at what `.describe()` gives you. This is often the
first thing a data scientist runs.


In [ ]:
print(df_clean['field_strength'].describe())


### Scatter plots — which features look related to the target?

Signal strength (dBu) has to depend on *something*. Distance is the obvious guess, but
what about power (`erp`) or antenna height (`haat`)? One `sns.scatterplot` per feature:


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.scatterplot(data=df_clean, x='distance', y='field_strength', ax=axes[0], alpha=0.3, s=15)
axes[0].set_title('Distance vs Signal')

sns.scatterplot(data=df_clean, x='erp', y='field_strength', ax=axes[1], alpha=0.3, s=15)
axes[1].set_title('ERP (Power) vs Signal')

sns.scatterplot(data=df_clean, x='haat', y='field_strength', ax=axes[2], alpha=0.3, s=15)
axes[2].set_title('HAAT (Antenna Height) vs Signal')

plt.tight_layout()
plt.show()


### The big one — a pairplot of everything at once

`sns.pairplot` draws a scatter plot for every pair of columns, with a histogram on the diagonal.
This is a classic "first look" for ML — one chart to spot which features actually matter.

*This cell takes 10–20 seconds to render. Patience.*


In [ ]:
sns.pairplot(
    df_clean,
    vars=['distance', 'erp', 'haat', 'frequency', 'field_strength'],
    diag_kind='hist',
    plot_kws={'alpha': 0.3, 's': 8},
    height=1.8,
)
plt.show()


**Predict before you model.** Look at the bottom row (everything vs `field_strength`).
- Which feature shows the clearest relationship with signal strength?
- Which one looks like it might not help at all?
- Based on just these pictures — what R² would you guess a distance-only model gets?
  **0.2? 0.5? 0.8?** Write down your guess. We'll find out in Part 1.


---
## Part 1 — Model 1: Distance Only

Simplest possible model: predict signal strength from distance alone. Physics says farther =
weaker, so there should be *some* signal. But how much? That's what R² tells us.

**Regression** = predicting a continuous number (not a category). `LinearRegression` finds
the best-fit line through the training data by minimizing total squared error.

**Why split train/test?** If you score a model on the same data it trained on, you're
measuring memorization, not learning. We hold out 20% as a test set and score on that.

### The 5-step workflow — same every time:
1. Choose features `X` and target `y`
2. Split into train / test sets
3. **Fit** the model on training data (`.fit()`)
4. **Predict** on test data (`.predict()`)
5. **Evaluate** — how close were the predictions?


### Step 1 — Choose features (X) and target (y)

*Note on the double brackets:* `df[['distance']]` is a 2D table (what scikit-learn wants for X).
`df['distance']` is 1D — fine for y, not for X.


In [ ]:
# Step 1 — Features and target
X1 = df_clean[['___']]       # <-- fill in: just distance
y  = df_clean['field_strength']

print(f"X shape: {X1.shape}")   # should be (1177, 1)
print(f"y shape: {y.shape}")    # should be (1177,)


### Step 2 — Split into train and test sets

Before the model sees any data, we set aside 20% of it as a **test set** — data the model will
never train on. After training, we score it on that hidden 20% to see how well it generalizes
to stations it has never seen.

`random_state=42` just fixes the random shuffle so everyone in class gets the same split.
The four variables you name here will be used in every step that follows.


In [ ]:
# Step 2 — Train / test split (80% train, 20% test)
# Name your four variables: X1_train, X1_test, y1_train, y1_test
___, ___, ___, ___ = train_test_split(X1, y, test_size=0.2, random_state=42)

print(f"Training rows : {len(X1_train)}")
print(f"Test rows     : {len(X1_test)}")


### Step 3 — Fit the model

`.fit()` is where the learning happens — it finds the slope and intercept that minimize
error on the training data. After fitting:

- `model.coef_[0]` → slope. Should be **negative** (farther = weaker).
- `model.intercept_` → y-intercept (predicted field strength at distance 0).


In [ ]:
# Step 3 — Create and fit the model
model1 = LinearRegression()
model1.fit(___, ___)   # <-- fill in: X1_train, y1_train

coef = model1.coef_[0]
intercept = model1.intercept_
print(f"Learned equation: field_strength = {coef:.4f} × distance + {intercept:.2f}")


### Steps 4 & 5 — Predict, then evaluate

Run `.predict()` on the test features, then score with two metrics:

- **R²** → 0 to 1. Fraction of variation the model explains. R² = 0.48 ⇒ model explains 48%.
- **RMSE** → average error in the target's units (dBu). RMSE = 16 ⇒ off by ~16 dBu per prediction.

**Before you run the next cell — write down your guess.** Will R² land closer to **0.2, 0.5, or 0.8**?


In [ ]:
# Step 4 — Predict on the test set
y_pred1 = model1.predict(___)   # <-- fill in: X1_test

# Step 5 — Evaluate
r2_1   = r2_score(y1_test, y_pred1)
rmse_1 = np.sqrt(mean_squared_error(y1_test, y_pred1))

print(f"R²   : {r2_1:.3f}")
print(f"RMSE : {rmse_1:.1f} dBu")


In [ ]:
# Scatter: actual vs predicted
plt.figure(figsize=(6, 5))
plt.scatter(y1_test, y_pred1, alpha=0.4, s=15, color='steelblue')
plt.plot([y1_test.min(), y1_test.max()],
         [y1_test.min(), y1_test.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Field Strength (dBu)')
plt.ylabel('Predicted Field Strength (dBu)')
plt.title('Model 1: Distance Only')
plt.legend()
plt.tight_layout()
plt.show()


**Q1 — Interpreting Model 1**

R² ≈ 0.48, RMSE ≈ 16.4 dBu. Did it match your guess from the pairplot? The distance coefficient
is negative — does that make physical sense? What else might explain the other 52%?

*(Double-click to answer.)*

> **Your answer:**


---
## Part 2 — Model 2: Add Power, Frequency, and Height

Distance alone explained ~48% of the variation. What's in the other 52%?

A 100 kW transmitter obviously reaches farther than a 1 kW one.
An antenna mounted on a mountain clears terrain that a rooftop antenna can't.
Higher frequencies behave differently than lower ones.

Let's add three more features the FCC measures for every licensed station:

| Feature | Column | What it means |
|---|---|---|
| Effective Radiated Power | `erp` | How much power the transmitter puts out (kW) |
| FM Frequency | `frequency` | The dial position in MHz (e.g., 93.5) |
| Height Above Average Terrain | `haat` | How high the antenna is above surrounding terrain (meters) |

Adding features to a linear model doesn't require any structural change —
just pass a wider `X` matrix. The model finds the best coefficient for each one.


In [ ]:
# Step 1 — Four features this time
FEATURES2 = ['___', '___', '___', '___']   # distance, erp, frequency, haat

X2 = df_clean[FEATURES2]

# Steps 2–4 — split, fit, predict
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.2, random_state=42)

model2 = LinearRegression()
model2.fit(X2_train, y2_train)
y_pred2 = model2.predict(X2_test)

r2_2   = r2_score(y2_test, y_pred2)
rmse_2 = np.sqrt(mean_squared_error(y2_test, y_pred2))
print(f"R²   : {r2_2:.3f}")
print(f"RMSE : {rmse_2:.1f} dBu")


In [ ]:
# What did the model learn? — feature coefficients
coef_df = pd.DataFrame({
    'feature':     FEATURES2,
    'coefficient': model2.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print(coef_df.to_string(index=False))


### Reading the coefficient table

Each coefficient tells you: *"for every one-unit increase in this feature,
how much does predicted field strength change, holding everything else constant?"*

- **Negative coefficient** → that feature weakens the signal as it increases
- **Positive coefficient** → that feature strengthens the signal as it increases
- **Larger absolute value** → that feature has more influence on the prediction

**Q2 — Which feature matters most?**

Do all four signs make physical sense? `distance` should be negative (farther = weaker).
What do the signs of `erp`, `haat`, and `frequency` tell you?
Which single feature has the largest effect on the prediction?

*(Double-click to answer.)*

> **Your answer:**


---
## Part 3 — Feature Engineering: Giving the Model Better Inputs

R² jumped from 0.48 → 0.74 by adding three features. Can we do better
without adding any new data columns?

**Feature engineering** means creating new columns derived from the ones you already have —
transforming the data into a shape the model can use more effectively.

---

### The Physics: Inverse Square Law

Radio signal power doesn't decrease linearly with distance — it decreases with the
**square** of distance. Double the distance, and the signal drops to **one quarter** strength:

```
Power ∝ 1 / distance²
```

This is a curved (non-linear) relationship. `LinearRegression` fits straight lines.
If the true relationship is curved, the model will never fit it well using raw distance.

**The fix:** take the log. `log(distance²)` = `2 × log(distance)`.
This converts the curve into a line — which a linear model can fit perfectly.

We do the same for ERP (power), which also has a non-linear relationship with field strength.
We also add `distance²` directly, giving the model the squared term explicitly.

Fill in the two `np.log()` formulas below. Add 1 inside the log to avoid `log(0)`.


In [ ]:
df_eng = df_clean.copy()

# Fill in the formulas
df_eng['log_distance'] = np.log(___ + 1)   # log transform of distance
df_eng['log_erp']      = np.log(___ + 1)   # log transform of erp
df_eng['distance_sq']  = df_eng['distance'] ** 2   # given — inverse square law

print("Sample engineered features:")
print(df_eng[['distance','log_distance','erp','log_erp','distance_sq']].head(4).to_string(index=False))


In [ ]:
# Train Model 3 with original + engineered features
FEATURES3 = ['distance', 'erp', 'frequency', 'haat',
             'log_distance', 'log_erp', 'distance_sq']

X3 = df_eng[FEATURES3]
y3 = df_eng['field_strength']

X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.2, random_state=42)

model3 = LinearRegression()
model3.fit(X3_train, y3_train)
y_pred3 = model3.predict(X3_test)

r2_3   = r2_score(y3_test, y_pred3)
rmse_3 = np.sqrt(mean_squared_error(y3_test, y_pred3))
print(f"R²   : {r2_3:.3f}")
print(f"RMSE : {rmse_3:.1f} dBu")


### Switching models is one line

Linear regression needed us to hand-engineer the inverse-square relationship for it.
What if we used a model that handles non-linearity automatically?

**Random Forest** is an ensemble of decision trees. It doesn't need log transforms —
it can learn curves directly from the raw features. Watch what happens when we swap
`LinearRegression()` for `RandomForestRegressor()`, using the same 4 raw features from Model 2.

**Predict before running:** how much higher will R² go? Will it clear 0.90?


In [ ]:
# Model 4 — Random Forest on the same 4 raw features as Model 2
# (no log transforms, no distance_sq — the forest doesn't need them)
model4 = RandomForestRegressor(n_estimators=100, random_state=42)
model4.fit(X2_train, y2_train)
y_pred4 = model4.predict(X2_test)

r2_4   = r2_score(y2_test, y_pred4)
rmse_4 = np.sqrt(mean_squared_error(y2_test, y_pred4))
print(f"R²   : {r2_4:.3f}")
print(f"RMSE : {rmse_4:.1f} dBu")


**Notice:** we changed exactly one line (`LinearRegression()` → `RandomForestRegressor(...)`).
The fit/predict/evaluate scaffold is identical. Once you know this workflow, you can swap in
XGBoost, gradient boosting, neural networks — any sklearn-style regressor works the same way.


In [ ]:
# Compare all four models — given, just run it
results = pd.DataFrame({
    'Model':    ['1: Distance only', '2: + Power/Freq/Height', '3: + Log transforms', '4: Random Forest'],
    'Features': [1, 4, 7, 4],
    'R²':       [r2_1, r2_2, r2_3, r2_4],
    'RMSE (dBu)': [rmse_1, rmse_2, rmse_3, rmse_4],
})
print(results.to_string(index=False))

plt.figure(figsize=(8, 4))
bars = plt.bar(results['Model'], results['R²'], color=['#aec6cf','#5b92b5','#1b3a6b','#2e7d32'])
plt.ylabel('R² Score')
plt.title('Model Comparison — R² by Approach')
plt.ylim(0, 1)
for bar, val in zip(bars, results['R²']):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.2f}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.xticks(rotation=10, ha='right')
plt.tight_layout()
plt.show()


**Q3 — The improvement arc**

Why did log transforms help the linear model so much, but the Random Forest didn't need them?
What does that tell you about when to use which type of model?

*(Double-click to answer.)*

> **Your answer:**


---
## Part 4 — Where Does the Model Go Wrong?

We have 25 receiver cities in this dataset. Instead of me handing you one to look at,
let's let the data pick. Score Model 3 city-by-city and rank by RMSE — then map the
**worst-performing** city to see what the model is missing.


### Step 1 — Rank all 25 cities by model error

The helper below computes RMSE for one city. Use it to build a table across all cities,
then sort to see which ones the model struggles with most.


In [ ]:
def city_rmse(city):
    sub = df_eng[df_eng['search_location'] == city]
    if len(sub) == 0:
        return None, 0
    pred = model3.predict(sub[FEATURES3])
    rmse = np.sqrt(((pred - sub['field_strength']) ** 2).mean())
    return rmse, len(sub)

rows = []
for c in df_eng['search_location'].unique():
    rmse, n = city_rmse(c)
    rows.append({'city': c, 'n_stations': n, 'rmse_dbu': rmse})

city_results = pd.DataFrame(rows).sort_values('rmse_dbu', ascending=False).reset_index(drop=True)
print(city_results.to_string(index=False))


### Step 2 — Visualize the spread

A bar chart makes the ranking easier to read than a 25-row table. Which cities stand out?


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=city_results, x='rmse_dbu', y='city', color='steelblue')
plt.xlabel('RMSE (dBu)')
plt.ylabel('')
plt.title('Model 3 Error by Receiver City')
plt.tight_layout()
plt.show()


### Step 3 — Map the worst-performing city

Pick the top of the table — the city with the highest RMSE — and map every station
it receives. Color by whether the model over- or under-predicted:

- **Green** → within 5 dBu (model nailed it)
- **Orange** → over-predicted by more than 5 dBu (model too optimistic)
- **Red** → under-predicted by more than 5 dBu (model too pessimistic)

Replace `CITY` with the worst city from your ranked table.


In [ ]:
CITY = '___'   # <-- replace with the worst-performing city from Step 1

city_df = df_eng[df_eng['search_location'] == CITY].copy()
city_df['predicted'] = model3.predict(city_df[FEATURES3])
city_df['error'] = city_df['predicted'] - city_df['field_strength']

# Center the map on the receiver location
center_lat = city_df['search_lat'].iloc[0]
center_lon = city_df['search_lon'].iloc[0]

m = folium.Map(location=[center_lat, center_lon], zoom_start=8)

# Mark the receiver itself
folium.Marker([center_lat, center_lon], tooltip=f'Receiver: {CITY}',
              icon=folium.Icon(color='blue', icon='info-sign')).add_to(m)

for _, row in city_df.iterrows():
    if abs(row['error']) <= 5:
        color = 'green'
    elif row['error'] > 5:
        color = 'orange'
    else:
        color = 'red'

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color=color,
        fill=True,
        fill_opacity=0.75,
        tooltip=(
            f"{row['callsign']} ({row['frequency']} MHz)<br>"
            f"Actual: {row['field_strength']:.1f} dBu<br>"
            f"Predicted: {row['predicted']:.1f} dBu<br>"
            f"Error: {row['error']:+.1f} dBu"
        )
    ).add_to(m)

print(f"{CITY}: {len(city_df)} stations, RMSE {np.sqrt((city_df['error']**2).mean()):.1f} dBu")
display(m)


**Q4 — What is the model missing?**

Look at your worst city on the map. Are the red/orange dots clustered in a direction
(say, all to the west), or scattered randomly? What geography might explain it —
mountains, coastline, a dense urban core? What information would the model need to do
better, that isn't in the FCC license data?

*(Double-click to answer.)*

> **Your answer:**


---
## Stretch — Compare the Best vs the Worst

Now pick the **best-performing** city (bottom of your ranked table) and map it the same way.
Does the error pattern look different? If there's a geography-based story in your worst
city, does the best city tell the opposite story?

*Copy the map code above into the cell below and change `CITY`.*


In [ ]:
# Stretch — the best-performing city
CITY = '___'

# YOUR CODE HERE (copy from Step 3 above)


---
## Q5 — So What? The Engineering Question

You've built a model that predicts FM signal strength with an average error of ~7.5 dBu
using nothing but publicly available FCC license data.

Think about that from a broadcast engineer's perspective:

- **Coverage planning:** A station applying for a license or a power increase needs to
  demonstrate it won't interfere with existing stations. Could a model like this help
  estimate coverage contours before filing with the FCC?
- **Interference analysis:** If a new station is proposed in your market, could you use
  a model like this to predict how much it would affect your signal at key locations?
- **Missing features:** What does a real broadcast engineer know about a signal that
  this model doesn't? (Think: terrain between transmitter and receiver, antenna pattern,
  building penetration, receiver quality.)

**In 3–5 sentences:** If you handed this model to a radio station engineer, what is one
specific decision it could help them make — and what would it get wrong?

*(Double-click to answer.)*

> **Your answer:**


---
## Part 5 — Stretch: Visualize Your FM Dial

Let's take advantage once more of Python's graphing capabilities — this time to draw
something that looks a lot like a car radio. Turn the knob from 88 to 108 MHz: some
positions light up with a strong station, some are static. Put frequency on the x-axis
and field strength on the y-axis, and every receivable station becomes one dot.


### Step 1 — Pick a receiver city

Any of the 25 cities from Part 4 works. Here's the list again:


In [ ]:
sorted(df_eng['search_location'].unique())


### Step 2 — Draw the dial

The plot below encodes **three** things at once for every station it receives:

- **x-axis:** frequency (88–108 MHz) — the position on the dial
- **y-axis:** field strength (dBu) — how strong it comes in
- **dot size:** ERP — bigger dot = more powerful transmitter
- **dot color:** distance — yellow = nearby, dark purple = far

We also label the 8 strongest stations with their callsigns, and draw a dashed line at
30 dBu as a rough "below this is static" reference.


In [ ]:
CITY = '___'   # <-- pick any city from the list above

dial = df_eng[df_eng['search_location'] == CITY].copy()
print(f"{CITY}: {len(dial)} stations receivable")

plt.figure(figsize=(14, 6))

sc = plt.scatter(
    dial['frequency'],
    dial['field_strength'],
    s=dial['erp'] * 3 + 20,       # size by ERP (power)
    c=dial['distance'],           # color by distance
    cmap='viridis_r',             # near = yellow, far = purple
    alpha=0.7,
    edgecolors='white',
    linewidths=0.8,
)
plt.colorbar(sc, label='Distance to transmitter (km)')

# Label the 8 strongest stations
top = dial.nlargest(8, 'field_strength')
for _, r in top.iterrows():
    plt.annotate(
        r['callsign'],
        xy=(r['frequency'], r['field_strength']),
        xytext=(5, 5), textcoords='offset points',
        fontsize=9, fontweight='bold',
    )

plt.axhline(y=30, color='red', linestyle='--', alpha=0.4,
            label='~30 dBu — rough noise floor')
plt.xlim(87.5, 108.5)
plt.xlabel('Frequency (MHz)')
plt.ylabel('Field Strength (dBu)')
plt.title(f'The FM Dial — What {CITY} Actually Hears')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


**Q6 — Read your dial**

Three things to notice on *your* city's plot (answers will differ city to city):

1. **The top tier.** Several of the strongest stations probably sit at almost the exact
   same field strength — a near-horizontal line across the top. Check their distance color.
   What does that tell you about where those transmitters physically are?
2. **The gaps.** Where on the dial is there *nothing*? In the US, 88.1–91.9 MHz is reserved
   for non-commercial/public radio. Does your city's lower-dial look sparser than the upper?
3. **The outlier.** Find a station that's **far** (dark color) but **strong** (near the top).
   Check its dot size. What makes it punch above its distance?

*(Double-click to answer.)*

> **Your answer:**


### Stretch — Two dials, side by side

Pick two cities you expect to feel different — say a dense market (Los Angeles, Atlanta)
vs a sparse one (Bangor, Anchorage, Key West, Cheyenne) — and plot their dials on top of
each other. Which dial is more crowded? Which has more empty space?


In [ ]:
CITY_A = '___'
CITY_B = '___'

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, city in zip(axes, [CITY_A, CITY_B]):
    d = df_eng[df_eng['search_location'] == city]
    ax.scatter(d['frequency'], d['field_strength'],
               s=d['erp']*3 + 20, c=d['distance'], cmap='viridis_r',
               alpha=0.7, edgecolors='white', linewidths=0.8)
    ax.axhline(y=30, color='red', linestyle='--', alpha=0.4)
    ax.set_title(f'{city} — {len(d)} stations')
    ax.set_ylabel('dBu')
    ax.grid(True, alpha=0.3)

axes[1].set_xlabel('Frequency (MHz)')
axes[1].set_xlim(87.5, 108.5)
plt.tight_layout()
plt.show()
